<center><h1>Park_Peter_HW3</h1></center>
<br>
<br>

Name: Peter Park
<br>
Github Username: pspark1052
<br>
USC ID: 4240400712

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download Data

Package imports

In [1]:
import os
import numpy as np
import pandas as pd

Get the AReM Data Set

In [2]:
activities = ["bending1", "bending2", "cycling", "lying", "sitting", "standing", "walking"]
columns = ["time", "avg_rss12", "var_rss12", "avg_rss13", "var_rss13", "avg_rss23", "var_rss23"]

instances = []

for activity in activities:
    for num in range(1, 16):
        path = "Data/AReM/" + activity + "/dataset" + str(num) + ".csv"
        if os.path.exists(path):
            df = pd.read_csv(path, comment="#", header=None, sep=r"[,\s]+",
                             engine="python", usecols=range(7), names=columns)
            instances.append([activity, num, df])

print(len(instances))

88


### (b) Test and Train Data

In [3]:
train = []
test = []

for inst in instances:
    activity, num, df = inst
    if activity in ["bending1", "bending2"] and num <= 2:
        test.append(inst)
    elif activity not in ["bending1", "bending2"] and num <= 3:
        test.append(inst)
    else:
        train.append(inst)

print("Train:", len(train))
print("Test:", len(test))

Train: 69
Test: 19


### (c) Feature Extraction

#### i. Research

- Minimum and maximum
- Mean and median
- Standard deviation and variance (how spread out the values are)
- Range (the max minus the min)
- First and third quartiles
- Skewness (whether the values lean to one side)
- Kurtosis (how many extreme values there are)
- Root mean square (how large the values are overall)
- Number of times the series crosses its own mean
- Autocorrelation (how much the series looks like a shifted copy of itself)

#### ii. Extraction

In [4]:
rows = []
for activity, num, df in instances:
    row = []
    for col in columns[1:]:
        s = df[col]
        row += [s.min(), s.max(), s.mean(), s.median(), s.std(), s.quantile(0.25), s.quantile(0.75)]
    rows.append(row)

names = []
for i in range(1, 7):
    names += [f"min{i}", f"max{i}", f"mean{i}", f"median{i}", f"std{i}", f"1st quart{i}", f"3rd quart{i}"]

features = pd.DataFrame(rows, columns=names, index=range(1, 89))
print(features.shape)
features

(88, 42)


,min1,max1,mean1,median1,std1,1st quart1,3rd quart1,min2,max2,mean2,...,std5,1st quart5,3rd quart5,min6,max6,mean6,median6,std6,1st quart6,3rd quart6
1,37.25,45.00,40.624792,40.50,1.476967,39.25,42.0000,0.0,1.30,0.358604,...,2.188449,33.0000,36.00,0.00,1.92,0.570583,0.43,0.582915,0.0000,1.3000
2,38.00,45.67,42.812812,42.50,1.435550,42.00,43.6700,0.0,1.22,0.372437,...,1.995255,32.0000,34.50,0.00,3.11,0.571083,0.43,0.601010,0.0000,1.3000
3,35.00,47.40,43.954500,44.33,1.558835,43.00,45.0000,0.0,1.70,0.426250,...,1.999604,35.3625,36.50,0.00,1.79,0.493292,0.43,0.513506,0.0000,0.9400
4,33.00,47.75,42.179812,43.50,3.670666,39.15,45.0000,0.0,3.00,0.696042,...,3.849448,30.4575,36.33,0.00,2.18,0.613521,0.50,0.524317,0.0000,1.0000
5,33.00,45.75,41.678063,41.75,2.243490,41.33,42.7500,0.0,2.83,0.535979,...,2.411026,28.4575,31.25,0.00,1.79,0.383292,0.43,0.389164,0.0000,0.5000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84,19.50,45.33,33.586875,34.25,4.650935,30.25,37.0000,0.0,14.67,4.576562,...,3.283983,13.7300,18.25,0.00,8.32,3.259729,3.11,1.640243,2.0500,4.3225
85,19.75,45.50,34.322750,35.25,4.752477,31.00,38.0000,0.0,13.47,4.456333,...,3.119856,13.5000,17.75,0.00,9.67,3.432563,3.20,1.732727,2.1575,4.5650
86,19.50,46.00,34.546229,35.25,4.842294,31.25,37.8125,0.0,12.47,4.371958,...,2.823124,14.0000,17.75,0.00,10.00,3.338125,3.08,1.656742,2.1600,4.3350
87,23.50,46.25,34.873229,35.25,4.531720,31.75,38.2500,0.0,14.82,4.380583,...,3.131076,13.7500,18.00,0.00,9.51,3.424646,3.27,1.690960,2.1700,4.5000


#### iii. Standard Deviation

In [5]:
np.random.seed(0)

results = []
for col in features.columns:
    boot = []
    for b in range(1000):
        boot.append(features[col].sample(88, replace=True).std())
    results.append([col, features[col].std(), np.percentile(boot, 5), np.percentile(boot, 95)])

ci = pd.DataFrame(results, columns=["Feature", "Std", "90% CI Lower", "90% CI Upper"])
ci

,Feature,Std,90% CI Lower,90% CI Upper
0,min1,9.569975,8.266015,10.783566
1,max1,4.394362,3.325273,5.280632
2,mean1,5.335718,4.705879,5.863331
3,median1,5.440054,4.777187,6.022142
4,std1,1.772153,1.572519,1.954187
5,1st quart1,6.153590,5.565187,6.626235
6,3rd quart1,5.138925,4.325072,5.836921
7,min2,0.000000,0.000000,0.000000
8,max2,5.062729,4.625818,5.393656
9,mean2,1.574164,1.389735,1.706690


#### iv. Select Features

I selected min, mean, and max. The mean shows the typical signal level, and the min and max show how far the signal moves, which should differ between active and still activities.

## 2. ISLR 3.7.4

### (a) Linear Train

The cubic model would have a lower training RSS. It is more flexible, so it can fit the random noise in the training data better.

### (b) Linear Test

The linear model would have a lower test RSS. The true relationship is linear, so the cubic model's extra terms just fit noise, which hurts it on new data.

### (c) Not Linear Train

The cubic model would have a lower training RSS. A more flexible model always fits the training data at least as well.

### (d) Not Linear Testing

There isn't enough information to tell. If the relationship is very curved, the cubic model would likely do better. If it is only slightly curved, the linear model could still do better.

## 3. ISLR 3.7.3 - Extra Practice 

## 3. ISLR 3.7.5 - Extra Practice 